In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, ConfusionMatrixDisplay)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from imblearn.over_sampling import SMOTE
import warnings; warnings.filterwarnings('ignore')

# Load dữ liệu
df = pd.read_csv('creditcard.csv')
print('Shape:', df.shape)
print('\nPhân bố nhãn:')
print(df['Class'].value_counts())
print('Tỷ lệ gian lận: {:.4f}%'.format(
    df['Class'].sum() / len(df) * 100))


In [ ]:
# Thống kê mô tả
print(df.describe())

# Kiểm tra missing values
print('Missing values:', df.isnull().sum().sum())

# Trực quan hóa phân bố lớp
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Biểu đồ cột
counts = df['Class'].value_counts()
axes[0].bar(['Bình thường (0)', 'Gian lận (1)'],
            counts.values, color=['steelblue', 'tomato'])
axes[0].set_title('Phân bố giao dịch')
axes[0].set_ylabel('Số lượng')

# Phân bố Amount theo lớp
df[df['Class'] == 0]['Amount'].hist(
    ax=axes[1], bins=50, alpha=0.6, label='Bình thường', color='steelblue')
df[df['Class'] == 1]['Amount'].hist(
    ax=axes[1], bins=50, alpha=0.8, label='Gian lận', color='tomato')
axes[1].set_title('Phân bố số tiền giao dịch')
axes[1].legend()
plt.tight_layout(); plt.show()


In [ ]:
# Chuẩn hóa Amount và Time
scaler = StandardScaler()
df['scaled_amount'] = scaler.fit_transform(df[['Amount']])
df['scaled_time']   = scaler.fit_transform(df[['Time']])
df.drop(['Amount', 'Time'], axis=1, inplace=True)

# Tách features và target
X = df.drop('Class', axis=1)
y = df['Class']

# Tách train / test (80 / 20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print('Train size:', X_train.shape)
print('Test size: ', X_test.shape)

# Áp dụng SMOTE trên tập train
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
print('\nSau SMOTE – phân bố y_train:')
print(pd.Series(y_train_sm).value_counts())


In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_sm, y_train_sm)
y_pred_lr = lr.predict(X_test)

print('=== LOGISTIC REGRESSION ===')
print(confusion_matrix(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr,
      target_names=['Bình thường', 'Gian lận']))


In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_sm, y_train_sm)
y_pred_rf = rf.predict(X_test)

print('=== RANDOM FOREST ===')
print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf,
      target_names=['Bình thường', 'Gian lận']))


In [ ]:
iso = IsolationForest(contamination=0.0017, random_state=42)
iso.fit(X_train)
y_raw = iso.predict(X_test)
y_pred_iso = [1 if v == -1 else 0 for v in y_raw]

print('=== ISOLATION FOREST ===')
print(confusion_matrix(y_test, y_pred_iso))
print(classification_report(y_test, y_pred_iso,
      target_names=['Bình thường', 'Gian lận']))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
models   = ['Logistic Regression', 'Random Forest', 'Isolation Forest']
preds    = [y_pred_lr, y_pred_rf, y_pred_iso]
colors   = ['Blues', 'Greens', 'Oranges']

for ax, name, pred, cmap in zip(axes, models, preds, colors):
    cm = confusion_matrix(y_test, pred)
    disp = ConfusionMatrixDisplay(cm,
               display_labels=['Bình thường', 'Gian lận'])
    disp.plot(ax=ax, colorbar=False, cmap=cmap)
    ax.set_title(name)

plt.suptitle('So sánh Confusion Matrix – 3 Mô hình', fontsize=14)
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
param_grid = {
    'n_estimators':     [100, 200, 300],
    'max_depth':        [10, 20, None],
    'min_samples_split': [2, 5, 10]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=3,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X_train_sm, y_train_sm)

print('Best params:', grid_search.best_params_)
print('Best F1    :', grid_search.best_score_)

# Đánh giá mô hình tối ưu
best_rf = grid_search.best_estimator_
y_pred_best = best_rf.predict(X_test)
print(classification_report(y_test, y_pred_best,
      target_names=['Bình thường', 'Gian lận']))


In [ ]:
importance = rf.feature_importances_
feat_df = pd.DataFrame({
    'feature':    X.columns,
    'importance': importance
}).sort_values('importance', ascending=False)

# In top 10
print(feat_df.head(10).to_string(index=False))

# Biểu đồ
plt.figure(figsize=(10, 6))
sns.barplot(data=feat_df.head(10), x='importance', y='feature',
            palette='viridis')
plt.title('Top 10 Feature Importance – Random Forest')
plt.xlabel('Mức độ quan trọng')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150)
plt.show()
